# 02_data_cleaning_analysis.ipynb
Enrich & clean + analysis

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [16]:
csv_path = '../data/raw/transactions_sample.csv'
df = pd.read_csv(csv_path)
df.head()

,order_id,date,category,product,quantity,price,sales
0,100001,2024-02-23,Clothing,T-Shirt,4,3496.22,13984.88
1,100002,2024-09-08,Groceries,Rice,4,349.78,1399.12
2,100003,2024-01-22,Clothing,Shoes,5,2489.97,12449.85
3,100004,2024-11-18,Electronics,Charger,6,18451.87,110711.22
4,100005,2024-07-04,Toys,Action Figure,4,2791.63,11166.52


In [17]:
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
df['revenue_per_unit'] = df['sales'] / df['quantity']
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['month'] = df['date'].dt.to_period('M').astype(str)
df = df[(df['quantity'] > 0) & (df['sales'] > 0)].copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 794 entries, 0 to 799
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          794 non-null    int64         
 1   date              794 non-null    datetime64[ns]
 2   category          774 non-null    object        
 3   product           778 non-null    object        
 4   quantity          794 non-null    int64         
 5   price             794 non-null    float64       
 6   sales             794 non-null    float64       
 7   revenue_per_unit  794 non-null    float64       
 8   month             794 non-null    object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 62.0+ KB


In [18]:
cat_avg_rev = df.groupby('category')['revenue_per_unit'].mean().sort_values(ascending=False)
cat_avg_rev

monthly = df.groupby(['category','month'])['sales'].sum().reset_index()
cat_vol = monthly.groupby('category')['sales'].std().sort_values(ascending=False)
cat_vol

prod_sales = df.groupby('product')['sales'].sum().sort_values(ascending=False)
prod_sales.head()

product
Smartwatch    1259788.51
Charger       1109828.50
Headphones     912604.99
Laptop         879658.27
Smartphone     373819.82
Name: sales, dtype: float64

In [19]:
plt.figure(figsize=(8,5))
sns.boxplot(x='category', y='revenue_per_unit', data=df)
plt.xticks(rotation=45)
plt.title('Revenue per Unit by Category')
plt.tight_layout()
plt.savefig('../reports/visuals/boxplot_revenue_per_unit_by_category.png')
plt.close()

In [20]:
top10 = df.groupby('product')['sales'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,6))
sns.barplot(x=top10.values, y=top10.index)
plt.title('Top 10 Products by Total Sales')
plt.xlabel('Total Sales')
plt.tight_layout()
plt.savefig('../reports/visuals/bar_top10_products_total_sales.png')
plt.close()

In [21]:
df.to_csv('../data/processed/transactions_cleaned.csv', index=False)
print('Saved cleaned dataset')

Saved cleaned dataset


## Notes
See reports/insights.md for summary.